In [ ]:
# Import required libraries for embedding, topic modeling, plotting, and persistence
import evoc

from sentence_transformers import SentenceTransformer
import datamapplot
import pandas as pd
import numpy as np
np.bool = np.bool_

import openai
import dill as pickle

from collections import Counter

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, OpenAI
from bertopic.dimensionality import BaseDimensionalityReduction

from umap import UMAP

In [ ]:
# Configure paths, column names, and model settings
project_path = ""

csv_data_file = f"{project_path}data_predicted_relevant.csv"

title_column = "title"
abstract_column = "abstract"

image_title = "Articles"

embedding_model_name = "intfloat/multilingual-e5-large-instruct"

In [ ]:
# Load data and keep only the relevant fields
data = pd.read_csv(csv_data_file)
data = data[["doi", title_column, abstract_column]]
data[title_column].fillna("", inplace=True)
data[abstract_column].fillna("", inplace=True)
data.reset_index(drop=True, inplace=True)

data

In [ ]:
# Create a document string for each row, combining title and abstract
docs = "Title: " + data["title"] + "\n\nAbstract: " + data["abstract"]
pd.options.display.max_colwidth = 200
docs

In [ ]:
# Load the sentence transformer model that produces document embeddings
embedding_model = SentenceTransformer(embedding_model_name)

In [ ]:
# Prepare OpenAI-based topic representation models
#representation_model = KeyBERTInspired()

key_proj_perovskita = "sk-XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"
client = openai.OpenAI(api_key=key_proj_perovskita)
oa = OpenAI(
    client,
    #model="llama-3.2-90b-vision-preview",
    model="gpt-4o-mini",
    delay_in_seconds=2,
    chat=True,
    nr_docs=8,
)

# Keyword-based and OpenAI-enhanced representation
kw = KeyBERTInspired()

main_representation_model = [kw, oa]

aspect1_model = [KeyBERTInspired()]

representation_model = {
   "Main": main_representation_model,
   "Aspect1":  aspect1_model,
}

# representation_model = [kw,oa]


In [ ]:
# Initialize BERTopic with EVoC clustering and custom representation
model = BERTopic(calculate_probabilities=True,
                 embedding_model = embedding_model,
                 umap_model=BaseDimensionalityReduction(),
                 hdbscan_model = evoc.EVoC(noise_level=0),
                 verbose=True,
                 representation_model = representation_model,
                )

In [ ]:
# Generate embeddings for all documents and save them for reuse
embs = embedding_model.encode(docs, show_progress_bar=True)
with open(f'{project_path}embs_pred.pkl', 'wb') as handle:
    pickle.dump(embs, handle, protocol=pickle.HIGHEST_PROTOCOL)

# with open(f'{project_path}/embs_pred.pkl', 'rb') as handle:
#     embs = pickle.load(handle)

In [ ]:
# Fit the BERTopic model using the precomputed embeddings
topics, probs = model.fit_transform(docs, embeddings=embs)

# with open(f'{project_path}topics_pred.pkl', 'wb') as handle:
#     pickle.dump(topics, handle, protocol=pickle.HIGHEST_PROTOCOL)
# with open(f'{project_path}probs_pred.pkl', 'wb') as handle:
#     pickle.dump(probs, handle, protocol=pickle.HIGHEST_PROTOCOL)
# model.save(f'{project_path}model_pred', serialization="safetensors", save_ctfidf=True, save_embedding_model=embedding_model)

# with open(f'{project_path}topics_pred.pkl', 'rb') as handle:
#     topics = pickle.load(handle)
# with open(f'{project_path}probs_pred.pkl', 'rb') as handle:
#     probs = pickle.load(handle)
# model = BERTopic.load(f'{project_path}model_pred')

In [ ]:
# Display topic metadata for the fitted model
model.get_topic_info()

In [ ]:
# Update topic representations for each clustering layer and collect labels
clusters_rep = []
clusters_name = []
topics = []
representation_set = []
names_set = []
keywords = []
for layer in range(len(model.hdbscan_model.cluster_layers_)):
    model.update_topics(docs=docs,
                        topics=model.hdbscan_model.cluster_layers_[layer],
                        representation_model=representation_model)
    labels_rep = [" ".join(label) for label in model.get_topic_info()['Representation']]
    all_labels_rep = [labels_rep[i+1] for i in model.hdbscan_model.cluster_layers_[layer]]
    labels_name = [" ".join(label) for label in model.get_topic_info()['Name']]
    all_labels_name = [labels_name[i+1] for i in model.hdbscan_model.cluster_layers_[layer]]
    clusters_rep.append(all_labels_rep)
    clusters_name.append(all_labels_name)
    topics.append(model.hdbscan_model.cluster_layers_[layer])
    representation_set.append(model.get_topic_info()['Representation'].to_list())
    names_set.append(model.get_topic_info()['Name'].to_list())
    keywords.append(model.get_topic_info()['Aspect1'].to_list())

In [ ]:
# Clean up cluster name formatting by removing extra spaces
for i in range(len(clusters_name)):
    clusters_name[i] = [x.replace("  ", " %%%%") for x in clusters_name[i]]
    clusters_name[i] = [x.replace(" ", "") for x in clusters_name[i]]
    clusters_name[i] = [x.replace("%%%%", " ") for x in clusters_name[i]]

clusters_name

In [ ]:
# Save clustering outputs for later use
with open(f'{project_path}/clusters_rep_pred.pkl', 'wb') as handle:
    pickle.dump(clusters_rep, handle, protocol=pickle.HIGHEST_PROTOCOL)
with open(f'{project_path}/clusters_name_pred.pkl', 'wb') as handle:
    pickle.dump(clusters_name, handle, protocol=pickle.HIGHEST_PROTOCOL)
with open(f'{project_path}/topics_pred.pkl', 'wb') as handle:
    pickle.dump(topics, handle, protocol=pickle.HIGHEST_PROTOCOL)
with open(f'{project_path}/representation_set_pred.pkl', 'wb') as handle:
    pickle.dump(representation_set, handle, protocol=pickle.HIGHEST_PROTOCOL)
with open(f'{project_path}/names_set_pred.pkl', 'wb') as handle:
    pickle.dump(names_set, handle, protocol=pickle.HIGHEST_PROTOCOL)
with open(f'{project_path}/keywords_pred.pkl', 'wb') as handle:
    pickle.dump(keywords, handle, protocol=pickle.HIGHEST_PROTOCOL)

# with open(f'{project_path}/clusters_rep_pred.pkl', 'rb') as handle:
#     clusters_rep = pickle.load(handle)
# with open(f'{project_path}/clusters_name_pred.pkl', 'rb') as handle:
#     clusters_name = pickle.load(handle)
# with open(f'{project_path}/topics_pred.pkl', 'rb') as handle:
#     topics = pickle.load(handle)
# with open(f'{project_path}/representation_set_pred.pkl', 'rb') as handle:
#     representation_set = pickle.load(handle)
# with open(f'{project_path}/names_set_pred.pkl', 'rb') as handle:
#     names_set = pickle.load(handle)
# with open(f'{project_path}/keywords_pred.pkl', 'rb') as handle:
#     keywords = pickle.load(handle)

In [ ]:
# Add topic labels from each layer back into the original data frame
for i in range(len(clusters_name)):
    data[f'topic_{i}'] = [t for t in topics[i]]
data

In [ ]:
# Reduce embeddings to 2D for plotting cluster points
reduced_embeddings = UMAP(
                            n_components=2  # maior gera mais clusters
                          ).fit_transform(embs)

In [ ]:
# Load saved cluster names and remove numeric prefixes from each label
with open(f'{project_path}/clusters_name_pred.pkl', 'rb') as handle:
    clusters_name = pickle.load(handle)

clusters_name = [[s.split('_')[1] for s in sublist] for sublist in clusters_name]
clusters_name

In [ ]:
# Format cluster labels and generate plots for each cluster
new_clusters_name = []

for sublist in clusters_name:
    counts = Counter(sublist)
    formatted_sublist = [
        f"{item} ({counts[item]})"
        .replace("Ni3S2","$\\mathrm{{Ni_3S_2}}$")
        .replace("MoS2","$\\mathrm{{MoS_2}}$")
        .replace("Co9S8","$\\mathrm{{Co_9S_8}}$")
        .replace("Co3O4","$\\mathrm{{Co_3O_4}}$")
        for item in sublist
    ]
    new_clusters_name.append(formatted_sublist)

for i, cluster_estudado in enumerate(new_clusters_name):
    image_title = "Papers Pedro Relevant"
    fig, ax = datamapplot.create_plot(
        reduced_embeddings,
        cluster_estudado,
        figsize=(11, 11),
        title=f"{image_title} - Cluster {i}",
        label_over_points=True,
        label_wrap_width=35, # 24,
        label_font_size=12,
        color_label_text=True, #False,
        min_font_weight=500,
        max_font_weight=501,
        font_family="Roboto Condensed"
    )
    fig.savefig(f"{project_path}{image_title}_cluster_{i}_v3.pdf", bbox_inches="tight")

In [ ]:
# Create Excel reports for each cluster with topic breakdowns and paper listings
cols = ['doi', 'title', 'abstract']
for i in range(len(clusters_name)):
    cols.append(f'topic_{i}')

tmp = data[cols]

for cluster_n, cluster in enumerate(clusters_name):
    print(f"Generating cluster {cluster_n}")
    names = names_set[cluster_n]
    names_temp = [(int(i[:i.index("_")]), i) for i in names]
    names_temp.sort(key=lambda tup: tup[0])
    cluster_kw = keywords[cluster_n]
    cluster_kw = [",".join(x) for x in cluster_kw]
    tmp[[f"topic_{cluster_n}_name"]] = ""
    writer = pd.ExcelWriter(f'{project_path}/papers_pred_cluster{cluster_n}.xlsx')
    df_temp = pd.DataFrame(Counter(cluster).items(), columns=[f'topics_cluster_{cluster_n}', 'count'])
    df_temp["keywords"] = cluster_kw
    df_temp["sort"] = df_temp[f'topics_cluster_{cluster_n}'].map(lambda x: int(x[:x.index("_")]))
    df_temp.sort_values(by=["sort"], inplace=True)
    df_temp.drop(columns=["sort"], inplace=True)
    df_temp.to_excel(writer, sheet_name="Topics", index=False)
    for i, grp in tmp.groupby(f'topic_{cluster_n}'):
        grp.to_excel(writer, sheet_name=names_temp[i+1][1][0:30].replace('/','_').replace(":","_"), index=False)
        tmp.loc[tmp[f"topic_{cluster_n}"] == grp.iloc[0][f"topic_{cluster_n}"], f"topic_{cluster_n}_name"] = names_temp[i+1][1]
    writer.close()